In [ ]:
# ============================================================
# NOTEBOOK — ρ* Threshold Validation
#            Tests nc/Ng at 5, 10, 12, 15, 20, 30, 50%
#            to empirically ground the ρ*≈0.10 claim
# Dataset: nazmusresan/fitzpatrick17k
# GPU: T4, Internet ON
# Runtime: ~60 min
# After running, paste ALL output back to Claude
# ============================================================

!pip install torch torchvision transformers scikit-learn \
             pandas numpy imbalanced-learn Pillow -q

import os, json, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')

# ── Config ────────────────────────────────────────────────────
NC_NG_TARGETS = [0.05, 0.10, 0.12, 0.15, 0.20, 0.30, 0.50]
SEEDS         = [42, 0, 1, 7, 99]
RANDOM_STATE  = 42
ETA_DRO       = 0.01

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
print(f'nc/Ng targets: {NC_NG_TARGETS}')
print(f'Seeds: {SEEDS}')

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def wilson_ci(k, n, z=1.96):
    if n == 0: return 0.0, 0.0
    p = k / n
    denom  = 1 + z**2 / n
    center = (p + z**2 / (2*n)) / denom
    margin = (z * np.sqrt(p*(1-p)/n + z**2/(4*n**2))) / denom
    return max(0.0, center - margin), min(1.0, center + margin)

# ── Dataset path auto-discovery ───────────────────────────────
_fitz_csv = None
for _root, _dirs, _files in os.walk('/kaggle/input'):
    for _f in _files:
        if _f.endswith('.csv') and 'fitzpatrick' in _f.lower():
            _fitz_csv = os.path.join(_root, _f)
            break
    if _fitz_csv:
        break

if _fitz_csv:
    FITZ_CSV     = _fitz_csv
    FITZ_IMG_DIR = os.path.dirname(_fitz_csv)
else:
    FITZ_CSV     = '/kaggle/input/fitzpatrick17k/fitzpatrick17k.csv'
    FITZ_IMG_DIR = '/kaggle/input/fitzpatrick17k/images'

print(f'CSV path : {FITZ_CSV}')
print(f'Image dir: {FITZ_IMG_DIR}')

# ── Load dataset ──────────────────────────────────────────────
print('\nLoading Fitzpatrick17k...')
df = pd.read_csv(FITZ_CSV)
df = df[df['fitzpatrick_scale'] > 0].copy()

image_files = {}
for _img_root, _img_dirs, _img_files in os.walk(FITZ_IMG_DIR):
    for _img_f in _img_files:
        if _img_f.endswith('.jpg') or _img_f.endswith('.png'):
            _key = _img_f.replace('.jpg', '').replace('.png', '')
            image_files[_key] = os.path.join(_img_root, _img_f)

print(f'Image files found: {len(image_files)}')
df['local_path'] = df['md5hash'].map(image_files)
df = df[df['local_path'].notna()].copy()
df['skin_group'] = df['fitzpatrick_scale'].apply(
    lambda x: 'light' if x <= 2 else 'medium' if x <= 4 else 'dark')

light_df  = df[df['skin_group'] == 'light'].sample(1000, random_state=RANDOM_STATE)
medium_df = df[df['skin_group'] == 'medium'].sample(1000, random_state=RANDOM_STATE)
dark_df   = df[df['skin_group'] == 'dark']

le = LabelEncoder()
le.fit(list(light_df['three_partition_label']) +
       list(medium_df['three_partition_label']) +
       list(dark_df['three_partition_label']))
print(f'Classes: {le.classes_}')
BENIGN_IDX = list(le.classes_).index('benign')

# ── Load CLIP ─────────────────────────────────────────────────
print('\nLoading CLIP ViT-L/14...')
clip_model = CLIPModel.from_pretrained('openai/clip-vit-large-patch14').to(device)
clip_proc  = CLIPProcessor.from_pretrained('openai/clip-vit-large-patch14')
clip_model.eval()
print('CLIP loaded.')

def load_imgs(dataframe):
    imgs, lbls = [], []
    for _, row in dataframe.iterrows():
        try:
            img = Image.open(row['local_path']).convert('RGB').resize((224, 224))
            imgs.append(img)
            lbls.append(le.transform([row['three_partition_label']])[0])
        except:
            pass
    return imgs, np.array(lbls)

@torch.no_grad()
def get_features(images, batch_size=32):
    all_feats = []
    for i in range(0, len(images), batch_size):
        batch  = images[i:i+batch_size]
        inputs = clip_proc(images=batch, return_tensors='pt', padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        # get_image_features may return a plain Tensor or a model output
        # object depending on transformers version — handle both
        out = clip_model.get_image_features(**inputs)
        if isinstance(out, torch.Tensor):
            feats = out
        elif hasattr(out, 'pooler_output') and out.pooler_output is not None:
            feats = out.pooler_output
        else:
            feats = out.last_hidden_state[:, 0]
        feats = feats.float()
        feats = feats / feats.norm(dim=-1, keepdim=True)
        all_feats.append(feats.cpu().numpy())
    return np.vstack(all_feats)

# ── Extract features ──────────────────────────────────────────
print('Loading images...')
light_imgs,  light_y  = load_imgs(light_df)
medium_imgs, medium_y = load_imgs(medium_df)
all_dark_df = dark_df.sample(min(2000, len(dark_df)), random_state=RANDOM_STATE)
dark_imgs,   dark_y   = load_imgs(all_dark_df)

print('Extracting features...')
light_feats  = get_features(light_imgs)
medium_feats = get_features(medium_imgs)
dark_feats   = get_features(dark_imgs)

# Fixed train base (light + medium) + fixed test set from dark
dark_idx = np.arange(len(dark_y))
pool_idx, test_idx = train_test_split(dark_idx, test_size=800,
                                       random_state=RANDOM_STATE,
                                       stratify=dark_y)
test_feats = dark_feats[test_idx]
test_y     = dark_y[test_idx]
pool_feats = dark_feats[pool_idx]
pool_y     = dark_y[pool_idx]

base_train_feats = np.vstack([light_feats, medium_feats])
base_train_y     = np.concatenate([light_y, medium_y])

n_dark_benign_pool = (pool_y == BENIGN_IDX).sum()
n_dark_benign_test = (test_y == BENIGN_IDX).sum()

print(f'\nPool: {len(pool_y)} dark | benign in pool: {n_dark_benign_pool}')
print(f'Test: {len(test_y)} dark | benign in test: {n_dark_benign_test}')

# ── nc/Ng pool builder ────────────────────────────────────────
def make_pool_at_nc_ng(target_nc_ng, total_pool_size=200, seed=RANDOM_STATE):
    np.random.seed(seed)
    target_n_benign = round(target_nc_ng * total_pool_size)
    target_n_other  = total_pool_size - target_n_benign

    benign_mask  = pool_y == BENIGN_IDX
    benign_feats = pool_feats[benign_mask]
    other_feats  = pool_feats[~benign_mask]
    benign_y_arr = pool_y[benign_mask]
    other_y_arr  = pool_y[~benign_mask]

    b_idx = np.random.choice(len(benign_feats), target_n_benign,
                              replace=(target_n_benign > len(benign_feats)))
    o_idx = np.random.choice(len(other_feats),  target_n_other,
                              replace=(target_n_other  > len(other_feats)))

    final_feats = np.vstack([benign_feats[b_idx], other_feats[o_idx]])
    final_y     = np.concatenate([benign_y_arr[b_idx], other_y_arr[o_idx]])

    perm = np.random.permutation(len(final_y))
    final_feats = final_feats[perm]
    final_y     = final_y[perm]

    actual_nc_ng = (final_y == BENIGN_IDX).sum() / len(final_y)
    return final_feats, final_y, actual_nc_ng

# ── Group DRO linear-probe ────────────────────────────────────
def run_group_dro_lp(seed, train_feats, train_y, pool_feats_nc, pool_y_nc,
                      eta=ETA_DRO, n_iter=100):
    np.random.seed(seed)
    groups = np.concatenate([
        np.zeros(len(train_y), dtype=int),
        np.ones(len(pool_y_nc), dtype=int)
    ])
    all_feats = np.vstack([train_feats, pool_feats_nc])
    all_y     = np.concatenate([train_y, pool_y_nc])
    n_groups      = 2
    group_weights = np.ones(n_groups) / n_groups
    clf = LogisticRegression(max_iter=500, C=1.0, random_state=seed)

    for _ in range(n_iter):
        clf.fit(all_feats, all_y, sample_weight=group_weights[groups])
        preds = clf.predict(all_feats)
        group_losses = np.array([
            1.0 - accuracy_score(all_y[groups == g], preds[groups == g])
            if (groups == g).sum() > 0 else 0.0
            for g in range(n_groups)
        ])
        group_weights = group_weights * np.exp(eta * group_losses)
        group_weights /= group_weights.sum()

    probs = clf.predict_proba(test_feats)
    preds = clf.predict(test_feats)
    try:
        auc = roc_auc_score(test_y, probs, multi_class='ovr', average='macro')
    except:
        auc = float('nan')
    benign_mask = test_y == BENIGN_IDX
    benign_acc  = float(accuracy_score(test_y[benign_mask], preds[benign_mask])) \
                  if benign_mask.sum() > 0 else 0.0
    final_min_wt = float(group_weights[1])
    return {
        'auc': float(auc),
        'benign_acc': benign_acc,
        'final_minority_weight': final_min_wt,
        'collapsed': final_min_wt < 0.01,
    }

# ── SMOTE linear-probe ────────────────────────────────────────
def run_smote_lp(seed, train_feats, train_y, pool_feats_nc, pool_y_nc):
    np.random.seed(seed)
    all_feats = np.vstack([train_feats, pool_feats_nc])
    all_y     = np.concatenate([train_y, pool_y_nc])
    try:
        k = max(1, min(5, (pool_y_nc == BENIGN_IDX).sum() - 1))
        aug_feats, aug_y = SMOTE(random_state=seed, k_neighbors=k).fit_resample(all_feats, all_y)
    except:
        aug_feats, aug_y = all_feats, all_y
    clf = LogisticRegression(max_iter=1000, C=1.0, random_state=seed)
    clf.fit(aug_feats, aug_y)
    probs = clf.predict_proba(test_feats)
    preds = clf.predict(test_feats)
    try:
        auc = roc_auc_score(test_y, probs, multi_class='ovr', average='macro')
    except:
        auc = float('nan')
    benign_mask = test_y == BENIGN_IDX
    benign_acc  = float(accuracy_score(test_y[benign_mask], preds[benign_mask])) \
                  if benign_mask.sum() > 0 else 0.0
    return {'auc': float(auc), 'benign_acc': benign_acc}

# ── Full sweep ────────────────────────────────────────────────
print('\n' + '='*60)
print('ρ* THRESHOLD SWEEP: Group DRO + SMOTE × nc/Ng × 5 seeds')
print('='*60)

all_results = {}

for nc_ng in NC_NG_TARGETS:
    print(f"\n{'─'*55}")
    print(f'nc/Ng = {nc_ng:.0%}')
    print(f"{'─'*55}")
    dro_results, smote_results = [], []

    for seed in SEEDS:
        pool_f, pool_y_nc, actual_nc_ng = make_pool_at_nc_ng(nc_ng, seed=seed)
        n_benign_in_pool = (pool_y_nc == BENIGN_IDX).sum()
        dro_r   = run_group_dro_lp(seed, base_train_feats, base_train_y, pool_f, pool_y_nc)
        smote_r = run_smote_lp(seed, base_train_feats, base_train_y, pool_f, pool_y_nc)
        dro_results.append(dro_r)
        smote_results.append(smote_r)
        print(f'  seed={seed}: DRO benign={dro_r["benign_acc"]:.4f} '
              f'(min_wt={dro_r["final_minority_weight"]:.3f}, '
              f'col={dro_r["collapsed"]})  '
              f'SMOTE benign={smote_r["benign_acc"]:.4f}  '
              f'pool_benign={n_benign_in_pool}/{len(pool_y_nc)}')

    dro_benigs   = [r['benign_acc'] for r in dro_results]
    smote_benigs = [r['benign_acc'] for r in smote_results]
    n_collapses  = sum(r['collapsed'] for r in dro_results)
    mean_dro     = np.mean(dro_benigs)
    std_dro      = np.std(dro_benigs)
    mean_smote   = np.mean(smote_benigs)
    std_smote    = np.std(smote_benigs)
    lo_dro, hi_dro = wilson_ci(round(mean_dro * n_dark_benign_test), n_dark_benign_test)

    print(f'\n  SUMMARY nc/Ng={nc_ng:.0%}:')
    print(f'    DRO   benign: {mean_dro:.4f} ± {std_dro:.4f}  '
          f'({lo_dro:.4f}–{hi_dro:.4f})  collapses={n_collapses}/{len(SEEDS)}')
    print(f'    SMOTE benign: {mean_smote:.4f} ± {std_smote:.4f}')

    all_results[f'{nc_ng:.2f}'] = {
        'nc_ng_target': nc_ng,
        'n_dark_benign_test': int(n_dark_benign_test),
        'dro': {
            'mean_benign': float(mean_dro), 'std_benign': float(std_dro),
            'wilson_ci': [float(lo_dro), float(hi_dro)],
            'n_collapsed': int(n_collapses),
            'per_seed': [{'seed': SEEDS[i],
                          'benign_acc': float(dro_results[i]['benign_acc']),
                          'collapsed': bool(dro_results[i]['collapsed'])}
                         for i in range(len(SEEDS))]
        },
        'smote': {
            'mean_benign': float(mean_smote), 'std_benign': float(std_smote),
            'per_seed': [{'seed': SEEDS[i],
                          'benign_acc': float(smote_results[i]['benign_acc'])}
                         for i in range(len(SEEDS))]
        }
    }

# ── Threshold analysis ────────────────────────────────────────
print('\n' + '='*60)
print('THRESHOLD TABLE: ρ* = nc/Ng below which DRO collapses')
print('='*60)
print(f"\n{'nc/Ng':>8} {'DRO Benign':>12} {'95% CI':>20} "
      f"{'Collapsed':>10} {'SMOTE Benign':>14}")
print('─'*70)
for nc_ng in NC_NG_TARGETS:
    r  = all_results[f'{nc_ng:.2f}']
    ci = r['dro']['wilson_ci']
    print(f"{nc_ng:>8.0%} {r['dro']['mean_benign']:>12.4f} "
          f"({ci[0]:.4f}–{ci[1]:.4f})   "
          f"{r['dro']['n_collapsed']:>4}/{len(SEEDS):<5} "
          f"{r['smote']['mean_benign']:>14.4f}")

threshold_candidates = [
    nc_ng for nc_ng in NC_NG_TARGETS
    if all_results[f'{nc_ng:.2f}']['dro']['mean_benign'] > 0.05
]
if threshold_candidates:
    print(f'\nEmpirical ρ* (lowest nc/Ng with DRO benign > 5%): {min(threshold_candidates):.0%}')
else:
    print('\nDRO fails to recover at ALL tested nc/Ng levels (benign <= 5% everywhere)')

print('\nCollapse rate progression:')
for nc_ng in NC_NG_TARGETS:
    r    = all_results[f'{nc_ng:.2f}']['dro']
    rate = r['n_collapsed'] / len(SEEDS)
    bar  = '#' * round(rate * 10)
    print(f'  nc/Ng={nc_ng:>5.0%}  collapses={r["n_collapsed"]}/{len(SEEDS)}  {bar}')

# ── Save ──────────────────────────────────────────────────────
output = {
    'experiment': 'rho_star_threshold_validation',
    'nc_ng_targets': NC_NG_TARGETS,
    'seeds': SEEDS,
    'eta_dro': ETA_DRO,
    'n_dark_benign_test': int(n_dark_benign_test),
    'results': all_results
}
with open('/kaggle/working/rho_star_threshold_validation.json', 'w') as f:
    json.dump(output, f, indent=2)

print('\nSaved: /kaggle/working/rho_star_threshold_validation.json')
print('\n Complete. Paste ALL output back to Claude.')